In [ ]:
# 1) Importar librerías
import os
import pandas as pd
import matplotlib.pyplot as plt

# Mostrar gráficos inline en Colab
%matplotlib inline

print('Librerías cargadas correctamente')

In [ ]:
# 2) Detectar archivos 'tienda_*.csv' en el directorio y unirlos en un solo DataFrame
archivos = [f for f in os.listdir('.') if f.lower().startswith('tienda') and f.lower().endswith('.csv')]

print('Archivos encontrados:', archivos)

if not archivos:
    raise FileNotFoundError('No se encontraron archivos tienda_*.csv. Sube los 4 CSV y vuelve a ejecutar.')

lista = []
for f in archivos:
    # Cargar cada CSV
    df_temp = pd.read_csv(f, encoding='utf-8', skipinitialspace=True)
    # Normalizar nombres de columnas 
    df_temp.columns = [c.strip().lower().replace(' ', '_').replace('\n','') for c in df_temp.columns]
    # Añadir columna 'tienda' con el nombre del archivO
    df_temp['tienda'] = os.path.splitext(f)[0]
    lista.append(df_temp)

# Unir todos los DataFrames
df = pd.concat(lista, ignore_index=True, sort=False)
print('\nDataset combinado - dimensiones:', df.shape)
print('\nEncabezados:') 
print(list(df.columns))
print('\nPrimeras filas:') 
df.head()

In [ ]:
# 3) Limpieza básica y conversión de tipos 
# Convertir precio y costo_de_envio a numérico si existen.
def limpiar_num(s):
    try:
        if pd.isna(s):
            return float('nan')
        t = str(s).replace('$','').replace('€','').replace(',','').strip()
        return float(t)
    except:
        return float('nan')

# Columnas esperadas en los CSV 
# producto, categoria_del_producto (o categoría), precio, costo_de_envio, fecha_de_compra, calificacion, lat, lon

# Intentar renombrar algunas columnas si tienen nombres diferentes
if 'categoría_del_producto' in df.columns and 'categoria_del_producto' not in df.columns:
    df = df.rename(columns={'categoría_del_producto':'categoria_del_producto'})

# Asegurar existencia de columnas clave
if 'precio' not in df.columns:
    raise KeyError('No se encontró la columna "Precio" en los CSV. Verifica los encabezados.')

# Aplicar limpieza numérica
df['precio'] = df['precio'].apply(limpiar_num)

if 'costo_de_envio' in df.columns:
    df['costo_de_envio'] = df['costo_de_envio'].apply(limpiar_num)
else:
    df['costo_de_envio'] = 0.0  # si no existe, asumir 0

# Calificación a numérica
if 'calificacion' in df.columns:
    df['calificacion'] = pd.to_numeric(df['calificacion'], errors='coerce')
else:
    df['calificacion'] = float('nan')

# Fecha a datetime si existe
if 'fecha_de_compra' in df.columns:
    df['fecha_de_compra'] = pd.to_datetime(df['fecha_de_compra'], errors='coerce')

print('Tipos después de limpieza:') 
print(df.dtypes)

In [ ]:
# 4) Métricas básicas por tienda (ingresos, costo de envío promedio, calificación promedio)
# Interpretación: cada fila es una venta => ingreso por fila = precio
df['ingreso'] = df['precio']

# Ingreso total por tienda
ingresos_tienda = df.groupby('tienda')['ingreso'].sum().reset_index().sort_values('ingreso', ascending=False)
print('\nIngreso total por tienda:')
print(ingresos_tienda)

# Costo de envío promedio por tienda
envio_prom_tienda = df.groupby('tienda')['costo_de_envio'].mean().reset_index().rename(columns={'costo_de_envio':'envio_promedio'})
print('\nCosto de envío promedio por tienda:')
print(envio_prom_tienda)

# Calificación promedio por tienda
calif_tienda = df.groupby('tienda')['calificacion'].mean().reset_index().rename(columns={'calificacion':'calificacion_promedio'})
print('\nCalificación promedio por tienda:')
print(calif_tienda)

# Unir resumen
resumen = ingresos_tienda.merge(envio_prom_tienda, on='tienda').merge(calif_tienda, on='tienda')
resumen = resumen.rename(columns={'ingreso':'ingreso_total'})
print('\nResumen combinado por tienda:')
print(resumen)

In [ ]:
# 5) Categorías más vendidas y productos más vendidos
# Cantidad de ventas por categoría (global y por tienda)
if 'categoria_del_producto' in df.columns:
    cats_global = df['categoria_del_producto'].value_counts().reset_index().rename(columns={'index':'categoria','categoria_del_producto':'cantidad'})
    print('\nTop categorías globales:') 
    print(cats_global.head(10))
else:
    print('\nNo se encontró columna "categoria_del_producto".')

# Productos más vendidos (conteo de filas por producto)
top_productos = df['producto'].value_counts().reset_index().rename(columns={'index':'producto','producto':'cantidad'})
print('\nTop productos globales:')
print(top_productos.head(10))

In [ ]:
# 6) Gráfico 1: Barras - Ingreso total por tienda
plt.figure(figsize=(8,5))
plt.bar(resumen['tienda'], resumen['ingreso_total'])
plt.title('Ingreso total por tienda')
plt.xlabel('Tienda')
plt.ylabel('Ingreso total')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 7) Gráfico 2: Pastel - Distribución de ventas por categoría (global)
if 'categoria_del_producto' in df.columns:
    top_cats = df['categoria_del_producto'].value_counts().head(6)
    plt.figure(figsize=(6,6))
    plt.pie(top_cats, labels=top_cats.index, autopct='%1.1f%%')
    plt.title('Distribución de ventas por categoría (top 6)')
    plt.show()
else:
    print('No hay columna de categoría para graficar.')

In [ ]:
# 8) Gráfico 3: Dispersión - Precio vs Calificación (muestra)
plt.figure(figsize=(8,5))
# Tomar datos no nulos
mask = df['precio'].notna() & df['calificacion'].notna()
plt.scatter(df.loc[mask,'precio'], df.loc[mask,'calificacion'], alpha=0.6)
plt.title('Precio vs Calificación de producto (ventas)')
plt.xlabel('Precio')
plt.ylabel('Calificación')
plt.ylim(0,5)
plt.grid(True)
plt.show()

In [ ]:
# 9) Recomendación final para el Sr. Juan (regla sencilla y explicada)
# Regla básica: elegir la tienda con menor ingreso total. Además se considera la calificación promedio.
resumen_sorted = resumen.sort_values('ingreso_total', ascending=True).reset_index(drop=True)
tienda_candidata = resumen_sorted.loc[0, 'tienda']
ingreso_cand = resumen_sorted.loc[0, 'ingreso_total']
calif_cand = resumen_sorted.loc[0, 'calificacion_promedio']

print(f'Tienda candidata a vender: {tienda_candidata}')
print(f'Ingreso total: {ingreso_cand}')
print(f'Calificación promedio: {calif_cand}')

# Texto de recomendación claro y en español
print('\nRECOMENDACIÓN (texto para el Sr. Juan):\n') 
print('Según los datos analizados (ingresos y calificaciones), la tienda que muestra menor desempeño es:', tienda_candidata + '.') 
print('Se recomienda considerar vender esta tienda para liberar capital y financiar el nuevo emprendimiento,') 
print('especialmente si además presenta costos fijos altos (alquiler, personal) o requiere inversiones para mejorar.')
print('\nAntes de venderla, sugerimos:') 
print('- Revisar los costos fijos y contratos de alquiler.')
print('- Analizar si la baja calificación se puede mejorar con acciones puntuales (mejorar atención o tiempos de envío).')
print('- Comparar margen (si se tienen datos de costos) para confirmar que la venta es la mejor opción.')